<a href="https://colab.research.google.com/github/MazurovaNN/right/blob/main/%D0%9C%D0%B0%D0%B7%D1%83%D1%80%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BE%D0%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install reportlab matplotlib

In [ ]:
font_path = "arial.ttf" # Будет искать прямо в папке со скриптом


In [ ]:
import os
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

def calculate_time_only(num_people, path_length_m, stairs_length_m, doors_widths_list):
    t_start = 0.5
    t_movement = (path_length_m / 60.0) + (stairs_length_m / 50.0)
    capacities = [50.0 * (w / 0.9) for w in doors_widths_list]
    min_capacity = min(capacities)
    t_door_delay = num_people / min_capacity
    return t_start + t_movement + t_door_delay

def generate_pdf_report(max_people, current_load, path_length_m, stairs_length_m, doors_widths_list, pdf_filename="Evacuation_Report.pdf"):

    # === ШАГ 1: АВТОНОМНЫЙ ПОИСК ЛОКАЛЬНОГО ШРИФТА ===
    # Составляем список стандартных путей к шрифту Arial в разных операционных системах
    possible_paths = [
        "C:\\Windows\\Fonts\\arial.ttf",       # Windows (основной путь)
        "C:\\Windows\\Fonts\\Arial.ttf",       # Windows (альтернативный регистр)
        "/Library/Fonts/Arial.ttf",            # macOS (старый путь)
        "/System/Library/Fonts/Supplemental/Arial.ttf", # macOS (современный путь)
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", # Linux / Ubuntu
        "arial.ttf"                            # Если вы вручную положили файл в папку
    ]

    font_path = None
    for path in possible_paths:
        if os.path.exists(path):
            font_path = path
            break

    if font_path:
        print(f"[ИНФО] Успешно найден локальный шрифт: {font_path}")
        pdfmetrics.registerFont(TTFont('LocalRussianFont', font_path))
        custom_font = 'LocalRussianFont'
    else:
        # Если система изолирована и шрифтов вообще нет (например, «голый» Docker-контейнер)
        print("[КРИТИЧЕСКАЯ ОШИБКА] Локальные шрифты с поддержкой русского языка не обнаружены!")
        print("РЕШЕНИЕ: Пожалуйста, вручную скопируйте любой файл шрифта (например, arial.ttf) ")
        print("из папки C:\\Windows\\Fonts на вашем компьютере и положите его ПРЯМО В ПАПКУ со скриптом.")
        return False

    # --- 2. РАСЧЕТЫ И МОДЕЛИРОВАНИЕ ---
    people_range = list(range(10, max_people + 1, 5))
    time_results = []
    t_required = 4.0
    critical_capacity = None

    for p in people_range:
        t_calculated = calculate_time_only(p, path_length_m, stairs_length_m, doors_widths_list)
        time_results.append(t_calculated)
        if t_calculated > t_required and critical_capacity is None:
            critical_capacity = p - 5

    if critical_capacity is None:
        min_cap = min([50.0 * (w / 0.9) for w in doors_widths_list])
        t_mov = (path_length_m / 60.0) + (stairs_length_m / 50.0)
        critical_capacity = int((t_required - 0.5 - t_mov) * min_cap)

    t_current = calculate_time_only(current_load, path_length_m, stairs_length_m, doors_widths_list)
    abs_deviation = t_current - t_required
    safety_margin_pct = (abs(abs_deviation) / t_required) * 100
    is_safe = t_current <= t_required

    # --- 3. ГЕНЕРАЦИЯ ГРАФИКА ---
    chart_filename = "temp_evacuation_chart.png"
    plt.figure(figsize=(8, 4.5))
    plt.plot(people_range, time_results, label="Расчетное время", color="#1f77b4", linewidth=2, marker='o', markersize=3)
    plt.axhline(y=t_required, color="#d62728", linestyle="--", linewidth=1.5, label=f"Лимит МЧС ({t_required} мин)")
    plt.title("Зависимость времени эвакуации от количества людей", fontsize=11, fontweight='bold')
    plt.xlabel("Количество людей (чел.)", fontsize=9)
    plt.ylabel("Время эвакуации (мин.)", fontsize=9)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=9, loc="upper left")
    plt.tight_layout()
    plt.savefig(chart_filename, dpi=300)
    plt.close()

    # --- 4. СБОРКА PDF ДОКУМЕНТА ---
    doc = SimpleDocTemplate(pdf_filename, pagesize=letter, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)
    styles = getSampleStyleSheet()

    style_title = ParagraphStyle('TitleStyle', parent=styles['Heading1'], fontName=custom_font, fontSize=16, leading=20, alignment=1, textColor=colors.HexColor("#1A365D"), spaceAfter=15)
    style_heading = ParagraphStyle('HeadingStyle', parent=styles['Heading2'], fontName=custom_font, fontSize=12, leading=16, textColor=colors.HexColor("#2C3E50"), spaceBefore=10, spaceAfter=8)
    style_body = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontName=custom_font, fontSize=10, leading=14, spaceAfter=6)
    style_callout = ParagraphStyle('CalloutStyle', parent=styles['Normal'], fontName=custom_font, fontSize=10, leading=14, textColor=colors.HexColor("#1E3A8A"))

    story = []

    story.append(Paragraph("ЭКСПЕРТНОЕ ЗАКЛЮЧЕНИЕ ПО РАСЧЕТУ ВРЕМЕНИ ЭВАКУАЦИИ", style_title))
    story.append(Paragraph("В рамках проверки требований антитеррористической защищенности объекта", style_body))
    story.append(Spacer(1, 10))

    # Таблица 1: Исходные данные
    story.append(Paragraph("1. Исходные параметры объекта", style_heading))
    data_inputs = [
        [Paragraph("<b>Параметр</b>", style_body), Paragraph("<b>Значение</b>", style_body), Paragraph("<b>Ед. изм.</b>", style_body)],
        [Paragraph("Проектная нагрузка (штат)", style_body), Paragraph(str(current_load), style_body), Paragraph("чел.", style_body)],
        [Paragraph("Общая длина коридоров", style_body), Paragraph(str(path_length_m), style_body), Paragraph("м.", style_body)],
        [Paragraph("Общая длина лестниц (вниз)", style_body), Paragraph(str(stairs_length_m), style_body), Paragraph("м.", style_body)],
        [Paragraph("Количество дверей на маршруте", style_body), Paragraph(str(len(doors_widths_list)), style_body), Paragraph("шт.", style_body)],
        [Paragraph("Ширина стандартного проема", style_body), Paragraph(f"{min(doors_widths_list):.2f}", style_body), Paragraph("м.", style_body)]
    ]
    t_inputs = Table(data_inputs, colWidths=[240, 140, 100])
    t_inputs.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#E2E8F0")),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#CBD5E1")),
        ('PADDING', (0,0), (-1,-1), 5),
    ]))
    story.append(t_inputs)
    story.append(Spacer(1, 15))

    # Таблица 2: Результаты хронометража
    story.append(Paragraph("2. Результаты расчета хронометража", style_heading))
    status_text = "ОБЪЕКТ СООТВЕТСТВУЕТ НОРМАМ" if is_safe else "ОПАСНОСТЬ! ПРЕВЫШЕНИЕ ЛИМИТА"

    data_results = [
        [Paragraph("<b>Этап эвакуации</b>", style_body), Paragraph("<b>Расчетное время</b>", style_body)],
        [Paragraph("Задержка начала эвакуации (t_нач)", style_body), Paragraph("0.50 мин.", style_body)],
        [Paragraph("Чистое движение по путям (t_дв)", style_body), Paragraph(f"{path_length_m/60.0 + stairs_length_m/50.0:.2f} мин.", style_body)],
        [Paragraph("Задержка в дверных проемах (t_инт)", style_body), Paragraph(f"{current_load / (50.0 * (min(doors_widths_list) / 0.9)):.2f} мин.", style_body)],
        [Paragraph("<b>Итоговое время эвакуации (t_расч)</b>", style_body), Paragraph(f"<b>{t_current:.2f} мин. ({int(t_current*60)} сек.)</b>", style_body)],
        [Paragraph("<b>Нормативный лимит МЧС (t_необх)</b>", style_body), Paragraph(f"<b>{t_required:.2f} min.</b>", style_body)],
        [Paragraph("<b>Статус соответствия нормам АТЗ</b>", style_body), Paragraph(f"<b>{status_text}</b>", style_body)]
    ]
    t_results = Table(data_results, colWidths=[300, 180])
    t_results.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#E2E8F0")),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#CBD5E1")),
        ('BACKGROUND', (0,-3), (-1,-2), colors.HexColor("#F8FAFC")),
        ('BACKGROUND', (0,-1), (-1,-1), colors.HexColor("#DCFCE7") if is_safe else colors.HexColor("#FEE2E2")),
        ('PADDING', (0,0), (-1,-1), 5),
    ]))
    story.append(t_results)
    story.append(Spacer(1, 15))

    # Добавление графика в PDF
    story.append(Paragraph("3. График зависимости и математическое моделирование", style_heading))
    story.append(Image(chart_filename, width=420, height=236))
    story.append(Spacer(1, 10))

    # Экспертное заключение
    story.append(Paragraph("4. Экспертное заключение комиссии", style_heading))
    if is_safe:
        conclusion_text = (f"Анализ отклонений показывает, что при проектной нагрузке в {current_load} чел. "
                           f"объект имеет чистый временной запас безопасности в размере {abs(abs_deviation):.2f} мин. "
                           f"({safety_margin_pct:.1f}% от нормы МЧС). Критическая точка вместимости эвакуационного "
                           f"маршрута составляет {critical_capacity} человек. Объект полностью удовлетворяет "
                           f"требованиям антитеррористической защищенности.")
    else:
        conclusion_text = (f"ВНИМАНИЕ! Проектная нагрузка в {current_load} чел. превышает пропускную способность "
                           f"дверных проемов. Превышение лимита составляет {abs_deviation:.2f} мин. ({safety_margin_pct:.1f}%). "
                           f"Безопасная эвакуация не гарантирована. Рекомендуется расширить критические створы дверей "
                           f"или рассредоточить людские потоки.")
    story.append(Paragraph(conclusion_text, style_body))

    doc.build(story)

    if os.path.exists(chart_filename):
        os.remove(chart_filename)

    print(f"[УСПЕХ] PDF-отчет успешно сформирован в автономном режиме: {pdf_filename}")

# --- ЗАПУСК МОДУЛЯ ---
generate_pdf_report(
    max_people=200,
    current_load=80,
    path_length_m=60,
    stairs_length_m=15,
    doors_widths_list=[1.0] * 8,
    pdf_filename="Evacuation_Report.pdf"
)


[ИНФО] Успешно найден локальный шрифт: arial.ttf
[УСПЕХ] PDF-отчет успешно сформирован в автономном режиме: Evacuation_Report.pdf


In [ ]:
import os
import matplotlib.pyplot as plt
import urllib.request
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

def calculate_time_only(num_people, path_length_m, stairs_length_m, doors_widths_list):
    t_start = 0.5
    t_movement = (path_length_m / 60.0) + (stairs_length_m / 50.0)
    min_capacity = min([50.0 * (w / 0.9) for w in doors_widths_list])
    return t_start + t_movement + (num_people / min_capacity)

def generate_official_pdf_report(max_people, current_load, path_length_m, stairs_length_m, doors_widths_list, pdf_filename="Official_Evacuation_Report.pdf"):
    # === ШАГ 1: ПОДКЛЮЧЕНИЕ ШРИФТА С ПОДДЕРЖКОЙ КИРИЛЛИЦЫ ===
    font_filename = "arial.ttf"
    if not os.path.exists(font_filename):
        font_filename = "C:\\Windows\\Fonts\\arial.ttf"
    if not os.path.exists(font_filename):
        font_filename = "Ubuntu-Regular.ttf"
        if not os.path.exists(font_filename):
            print("[ИНФО] Скачивание совместимого шрифта...")
            try:
                urllib.request.urlretrieve("https://githubusercontent.com", font_filename)
            except Exception as e:
                print(f"[ОШИБКА] Не удалось скачать шрифт: {e}")
                return False

    pdfmetrics.registerFont(TTFont('RussianFont', font_filename))
    custom_font = 'RussianFont'

    # --- ШАГ 2: МАТЕМАТИЧЕСКИЕ РАСЧЕТЫ ---
    people_range = list(range(10, max_people + 1, 5))
    time_results = [calculate_time_only(p, path_length_m, stairs_length_m, doors_widths_list) for p in people_range]
    t_required = 4.0

    critical_capacity = None
    for p, t in zip(people_range, time_results):
        if t > t_required and critical_capacity is None:
            critical_capacity = p - 5
    if critical_capacity is None:
        critical_capacity = max_people

    t_current = calculate_time_only(current_load, path_length_m, stairs_length_m, doors_widths_list)
    abs_deviation = t_current - t_required
    safety_margin_pct = (abs(abs_deviation) / t_required) * 100
    is_safe = t_current <= t_required

    # --- ШАГ 3: ГЕНЕРАЦИЯ ГРАФИКА ДЛЯ PDF ---
    chart_filename = "temp_evacuation_chart.png"
    plt.figure(figsize=(7, 3.8))
    plt.plot(people_range, time_results, label="Расчетное время", color="#1f77b4", linewidth=2)
    plt.axhline(y=t_required, color="#d62728", linestyle="--", label=f"Лимит МЧС ({t_required} мин)")
    plt.title("Зависимость времени эвакуации от количества людей", fontsize=10, fontweight='bold')
    plt.xlabel("Количество людей (чел.)", fontsize=8)
    plt.ylabel("Время эвакуации (мин.)", fontsize=8)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=8, loc="upper left")
    plt.tight_layout()
    plt.savefig(chart_filename, dpi=300)
    plt.close()

    # --- ШАГ 4: СБОРКА И ОФОРМЛЕНИЕ PDF ---
    doc = SimpleDocTemplate(pdf_filename, pagesize=letter, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)
    styles = getSampleStyleSheet()

    style_title = ParagraphStyle('Title', parent=styles['Heading1'], fontName=custom_font, fontSize=13, alignment=1, textColor=colors.HexColor("#1A365D"), spaceAfter=10)
    style_heading = ParagraphStyle('Heading', parent=styles['Heading2'], fontName=custom_font, fontSize=10, textColor=colors.HexColor("#2C3E50"), spaceBefore=10, spaceAfter=5)
    style_body = ParagraphStyle('Body', parent=styles['Normal'], fontName=custom_font, fontSize=9, leading=12)
    style_req = ParagraphStyle('Req', parent=styles['Normal'], fontName=custom_font, fontSize=8, leading=10, textColor=colors.HexColor("#475569"))

    story = []

    # Реквизиты
    org_info = "<b>ОГРАНИЗАЦИЯ:</b> ООО «БизнесЦентр-Развитие»<br/><b>Адрес:</b> г. Санкт-Петербург, пер. Эртелев, д. 12<br/><b>ИНН:</b> 7810123456 | <b>ОГРН:</b> 1237800987654"
    story.append(Paragraph(org_info, style_req))
    story.append(Spacer(1, 5))

    story.append(Paragraph("ПРИЛОЖЕНИЕ К ПАСПОРТУ БЕЗОПАСНОСТИ ОБЪЕКТА", style_title))
    story.append(Paragraph("<b>Технический отчет:</b> Расчет времени эвакуации при угрозе террористического акта.", style_body))
    story.append(Spacer(1, 10))

    # Таблица 1: Параметры
    story.append(Paragraph("1. Исходные параметры объекта", style_heading))
    data_inputs = [
        [Paragraph("<b>Наименование параметра</b>", style_body), Paragraph("<b>Значение</b>", style_body), Paragraph("<b>Ед. изм.</b>", style_body)],
        [Paragraph("Проектная численность людей на объекте", style_body), Paragraph(str(current_load), style_body), Paragraph("чел.", style_body)],
        [Paragraph("Общая протяженность путей до выхода", style_body), Paragraph(str(path_length_m + stairs_length_m), style_body), Paragraph("м.", style_body)],
        [Paragraph("Минимальная ширина створа дверей", style_body), Paragraph(f"{min(doors_widths_list):.2f}", style_body), Paragraph("м.", style_body)]
    ]
    t_inputs = Table(data_inputs, colWidths=[300, 100, 100])
    t_inputs.setStyle(TableStyle([('BACKGROUND', (0,0), (-1,0), colors.HexColor("#F1F5F9")), ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#E2E8F0")), ('PADDING', (0,0), (-1,-1), 4)]))
    story.append(t_inputs)
    story.append(Spacer(1, 10))

    # Таблица 2: Результаты
    story.append(Paragraph("2. Результаты моделирования хронометража", style_heading))
    status_text = "СООТВЕТСТВУЕТ ТРЕБОВАНИЯМ АТЗ" if is_safe else "НЕ СООТВЕТСТВУЕТ ТРЕБОВАНИЯМ!"

    data_results = [
        [Paragraph("<b>Наименование расчетного этапа</b>", style_body), Paragraph("<b>Время (мин.)</b>", style_body)],
        [Paragraph("Итоговое расчетное время полной эвакуации (t_расч)", style_body), Paragraph(f"<b>{t_current:.2f} мин.</b>", style_body)],
        [Paragraph("Критический предел времени по ГОСТ (t_необх)", style_body), Paragraph(f"<b>{t_required:.2f} мин.</b>", style_body)],
        [Paragraph("<b>Итоговое заключение по безопасности объекта</b>", style_body), Paragraph(f"<b>{status_text}</b>", style_body)]
    ]
    t_results = Table(data_results, colWidths=[350, 150])
    t_results.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#F1F5F9")),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#E2E8F0")),
        ('BACKGROUND', (0,-1), (-1,-1), colors.HexColor("#DCFCE7") if is_safe else colors.HexColor("#FEE2E2")),
        ('PADDING', (0,0), (-1,-1), 4)
    ]))
    story.append(t_results)
    story.append(Spacer(1, 10))

    # График
    story.append(Paragraph("3. Графический анализ пропускной способности путей", style_heading))
    story.append(Image(chart_filename, width=420, height=220))
    story.append(Spacer(1, 10))

    # Заключение
    story.append(Paragraph("4. Экспертные выводы комиссии", style_heading))
    if is_safe:
        conclusion_text = f"Установлено: при проектной нагрузке в {current_load} человек фактическое время эвакуации составляет {t_current:.2f} мин., что удовлетворяет требованиям МЧС. Временной запас составляет {abs(abs_deviation):.2f} мин. Критическая вместимость маршрута — {critical_capacity} человек. Объект признан безопасным."
    else:
        conclusion_text = f"ВНИМАНИЕ! Маршрут небезопасен для проектной нагрузки в {current_load} человек. Превышение лимита составляет {abs_deviation:.2f} мин. Вместимость объекта ограничена до {critical_capacity} человек."
    story.append(Paragraph(conclusion_text, style_body))
    story.append(Spacer(1, 15))

    # Форма подписей
    story.append(Paragraph("5. Лист согласования заключения", style_heading))
    story.append(Spacer(1, 5))

    sign_data = [
        [Paragraph("<b>СОГЛАСОВАНО</b><br/>Представитель ГУ МЧС России<br/>_______ / ___________ /<br/>«___» ________ 2026 г.", style_body), "",
         Paragraph("<b>СОГЛАСОВАНО</b><br/>Представитель УФСБ России<br/>_______ / ___________ /<br/>«___» ________ 2026 г.", style_body)],
        [Spacer(1, 15), "", Spacer(1, 15)],
        [Paragraph("<b>УТВЕРЖДАЮ</b><br/>Генеральный директор<br/>_______ / Иванов И.И. /<br/>«___» ________ 2026 г.", style_body), "",
         Paragraph("<b>ОТВЕТСТВЕННЫЙ ЗА АТЗ</b><br/>Специалист по безопасности<br/>_______ / Петров П.П. /<br/>«___» ________ 2026 г.", style_body)]
    ]

    t_signs = Table(sign_data, colWidths=[240, 20, 240])
    t_signs.setStyle(TableStyle([('VALIGN', (0,0), (-1,-1), 'TOP'), ('PADDING', (0,0), (-1,-1), 0)]))
    story.append(t_signs)

    doc.build(story)
    if os.path.exists(chart_filename):
        os.remove(chart_filename)
    print(f"[УСПЕХ] Официальный PDF-отчет сохранен: {pdf_filename}")

# --- ЗАПУСК ПОЛНОЙ СБОРКИ ---
generate_official_pdf_report(
    max_people=200,
    current_load=80,
    path_length_m=60,
    stairs_length_m=15,
    doors_widths_list=[1.0] * 8,
    pdf_filename="Official_Evacuation_Report.pdf"
)

[УСПЕХ] Официальный PDF-отчет сохранен: Official_Evacuation_Report.pdf
